In [4]:
import pandas as pd
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", None)

In [12]:
raw_dir = Path(".")
output_dir = Path("data")

output_dir.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv("Sales.csv", low_memory=False)
values = pd.read_csv("Values.csv", low_memory=False)
addresses = pd.read_csv("Owner_Address.csv", low_memory=False)

print("Sales records:", len(sales))
print("Value records:", len(values))
print("Address records:", len(addresses))

Sales records: 754401
Value records: 134871
Address records: 149213


In [16]:
sales_clean = sales.copy()

sales_clean["strap"] = sales_clean["strap"].astype("string").str.strip()

sales_clean["sale_date"] = pd.to_datetime(
    sales_clean["Tdate"],
    errors="coerce"
)

sales_clean["sale_year"] = sales_clean["sale_date"].dt.year

sales_clean["sale_price"] = pd.to_numeric(
    sales_clean["price"],
    errors="coerce"
)

sales_clean = sales_clean[
    sales_clean["sale_year"].isin([2019,2020,2026])
    &(sales_clean["sale_price"] > 0)
].copy()

sales_clean = sales_clean.drop_duplicates()

sales_clean = sales_clean[
    [
        "strap",
        "deedNum",
        "sale_date",
        "sale_year",
        "sale_price",
        "sales_cd",
        "deed_type"
    ]
]
sales_clean = sales_clean.rename(
    columns={"deedNum": "deed_number"}
)

print("Cleaned Sales:", len(sales_clean))
display(sales_clean.head(10))


/var/folders/h2/6mybx54539z1v31wl9yqffj80000gn/T/ipykernel_10602/237290948.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales_clean["sale_date"] = pd.to_datetime(


Cleaned Sales: 20926


,strap,deed_number,sale_date,sale_year,sale_price,sales_cd,deed_type
35,M2000013,3762532,2020-01-23,2020,45000.0,Q,VT
41,M2000015,3721944,2019-04-10,2019,56000.0,Q,VT
77,M2000031,4116816,2026-01-05,2026,95000.0,Q,VT
118,M2000045,3754826,2019-09-19,2019,29400.0,U,VT
145,M2000056,4124164,2026-03-10,2026,12500.0,Q,VT
176,M2000072,3726192,2019-07-25,2019,105000.0,Q,VT
200,M2000080,4152392,2026-03-15,2026,199000.0,P,VT
212,M2000083,3722560,2019-07-03,2019,69840.0,Q,VT
235,M2000089,4148381,2026-06-22,2026,39500.0,P,VT
257,M2000097,3710335,2019-05-01,2019,25000.0,Q,VT


In [17]:
values_clean = values.copy()

# Standardize property identifiers
values_clean["strap"] = (
    values_clean["strap"]
    .astype("string")
    .str.strip()
)

# Rename columns
values_clean = values_clean.rename(
    columns={
        "tax_yr": "assessment_year",
        "totalActualVal": "actual_value",
        "totalAssessedVal": "assessed_value"
    }
)

# Convert property values to numeric
for col in ["actual_value", "assessed_value"]:
    values_clean[col] = pd.to_numeric(
        values_clean[col],
        errors="coerce"
    )

# Keep relevant columns
values_clean = values_clean[
    [
        "strap",
        "assessment_year",
        "actual_value",
        "assessed_value"
    ]
]

# Remove duplicate property/year combinations
values_clean = values_clean.drop_duplicates(
    subset=["strap", "assessment_year"]
)

# Display results
print("Cleaned value records:", len(values_clean))
display(values_clean.head(10))

Cleaned value records: 134871


,strap,assessment_year,actual_value,assessed_value
0,M2000001,2026,92500,5661
1,M2000002,2026,55900,3421
2,M2000003,2026,39100,2393
3,M2000004,2026,66600,4076
4,M2000007,2026,70300,4302
5,M2000008,2026,85300,5220
6,M2000010,2026,56900,3482
7,M2000012,2026,47800,2925
8,M2000013,2026,41200,2521
9,M2000015,2026,26100,1775


In [19]:
addresses_clean = addresses.copy()

# Standardize property identifiers
addresses_clean["strap"] = (
    addresses_clean["strap"]
    .astype("string")
    .str.strip()
)

# Rename columns
addresses_clean = addresses_clean.rename(
    columns={
        "city": "property_city",
        "nh": "neighborhood",
        "account_type": "property_type"
    }
)

# Keep relevant columns
addresses_clean = addresses_clean[
    [
        "strap",
        "property_city",
        "neighborhood",
        "property_type"
    ]
]

# Remove duplicate property records
addresses_clean = addresses_clean.drop_duplicates(
    subset=["strap"]
)

# Display results
print("Cleaned address records:", len(addresses_clean))
display(addresses_clean.head(10))

Cleaned address records: 134746


,strap,property_city,neighborhood,property_type
0,M2000001,BOULDER,122.0,MANUFACTURED HOME
1,M2000002,BOULDER,122.0,MANUFACTURED HOME
2,M2000003,BOULDER,135.0,MANUFACTURED HOME
3,M2000004,BOULDER,122.0,MANUFACTURED HOME
4,M2000007,BOULDER,122.0,MANUFACTURED HOME
5,M2000008,LOUISVILLE,410.0,MANUFACTURED HOME
6,M2000010,BOULDER,122.0,MANUFACTURED HOME
8,M2000012,BOULDER,122.0,MANUFACTURED HOME
10,M2000013,BOULDER,122.0,MANUFACTURED HOME
11,M2000015,LAFAYETTE,460.0,MANUFACTURED HOME


In [20]:
# Keep the latest available assessment for each property
values_latest = (
    values_clean
    .sort_values("assessment_year")
    .drop_duplicates(subset=["strap"], keep="last")
)

# Merge sales with property values
df = sales_clean.merge(
    values_latest,
    on="strap",
    how="left",
    validate="many_to_one"
)

# Merge property location information
df = df.merge(
    addresses_clean,
    on="strap",
    how="left",
    validate="many_to_one"
)

# Sort records
df = df.sort_values(
    ["sale_year", "sale_date"]
).reset_index(drop=True)

# Check results
print("Original sales records:", len(sales_clean))
print("Combined records:", len(df))
print("Missing property values:", df["actual_value"].isna().sum())

display(df.head(10))

Original sales records: 20926
Combined records: 20926
Missing property values: 0


,strap,deed_number,sale_date,sale_year,sale_price,sales_cd,deed_type,assessment_year,actual_value,assessed_value,property_city,neighborhood,property_type
0,M9600064,3694398,2019-01-01,2019,38000.0,Q,VT,2026,22200,1510,LAFAYETTE,460.0,MANUFACTURED HOME
1,M8726247,3742326,2019-01-02,2019,3200.0,Q,VT,2026,12300,836,LAFAYETTE,460.0,MANUFACTURED HOME
2,M9900315,4009037,2019-01-02,2019,58900.0,U,VT,2026,46400,2840,LONGMONT,257.0,MANUFACTURED HOME
3,R0000367,3692340,2019-01-02,2019,500000.0,U,WD,2026,1213100,77731,BOULDER,101.0,RESIDENTIAL
4,R0011541,3693342,2019-01-02,2019,690000.0,Q,WD,2026,1025400,64967,BOULDER,150.0,RESIDENTIAL
5,R0011729,3694104,2019-01-02,2019,259000.0,U,SJ,2026,1061900,67449,BOULDER,150.0,RESIDENTIAL
6,R0031069,3692760,2019-01-02,2019,5700000.0,U,WD,2026,142700,37102,UNINCORPORATED,820.0,EXEMPT
7,R0040651,3692727,2019-01-02,2019,850000.0,U,WD,2026,1199100,299775,LONGMONT,298.0,COMMERCIAL
8,R0042153,3692851,2019-01-02,2019,260000.0,Q,RJ,2026,387800,23733,LONGMONT,204.0,RESIDENTIAL
9,R0056183,3692424,2019-01-02,2019,446000.0,Q,WD,2026,834300,51972,NEDERLAND,901.0,RESIDENTIAL


In [21]:
df.to_csv(
    raw_dir / "cleaned_full.csv",
    index = False
)